# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ssarimm/Machine-Learning-Internship---Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
month_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

q1 = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet('{month_path}')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

display(q1)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb

import duckdb
from google.colab import userdata

# Read the token from Colab Secrets; never print it.
hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{hf_token}')"
)

print("DuckDB is ready.")


DuckDB is ready.


In [2]:

month_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

schema = con.sql(
    f"DESCRIBE SELECT * FROM read_parquet('{month_path}')"
).df()

display(schema[["column_name", "column_type"]])


,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Research lane: Content Refresh Opportunity Scoring.

Raw data unit: One row represents the daily performance of one content item for one client on one reporting date (report_date, client_hash_id, content_hash_id).

Analysis window: March 1–31, 2026, using the month=2026-03 partition. The actual minimum and maximum dates will be verified by SQL.

Scoring unit: One row per client-content pair, aggregating its daily performance over March. This is the unit used to prioritize pages for human review.

Provisional prediction target: A future decline in page performance. The available March schema does not contain a confirmed decline label, so I will not pretend that one exists. A valid future-decline label would need to be defined using a later period, such as April, while keeping future data out of the March features.

Intended output: A ranked list of content pages for an SEO/content team to review—not an automatic decision to update every page.

**Q1 — Grain verification:** The duplicate-check query returned zero rows. This supports the claim that each combination of reporting date, client, and content item appears at most once in the March 2026 partition. The raw data grain is therefore consistent with one row per client-content-date combination.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Q2 — Row count and date span:** The March 2026 partition contains 9,841,378 daily performance records across 55 distinct clients and 331,437 distinct content IDs. The reporting dates range from March 1 through March 31, 2026, confirming that the partition covers the full calendar month. These are daily records, not yet a page-level monthly scoring dataset.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Q3 — GA4 availability:** Filtering the March 2026 partition with `ga4_data_available IS TRUE` retained 413,966 rows, covering 41 distinct clients and 90,489 distinct content IDs. These records represent approximately 4.21% of the partition's 9,841,378 rows. GA4 coverage is therefore limited in this slice, so GA4-derived features should be used cautiously and their availability explicitly considered.

In [5]:
q2 = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_content_ids,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet('{month_path}')
""").df()

display(q2)

,total_rows,unique_clients,unique_content_ids,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


In [7]:
display(q3.to_string(index=False))
print("Column names:", q3.columns.tolist())

' rows_surviving_ga4_filter  clients_with_available_ga4  content_ids_with_available_ga4\n                    413966                          41                           90489'

Column names: ['rows_surviving_ga4_filter', 'clients_with_available_ga4', 'content_ids_with_available_ga4']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data limits

- **Unbalanced history:** Clients have different amounts of historical data, so not every page can be compared over the same period. Missing historical records do not necessarily mean zero performance.
- **GSC-only early rows:** Some records may have Google Search Console (GSC) data but no Google Analytics 4 (GA4) data. Missing GA4 metrics must not be interpreted as zero traffic or zero engagement.
- **Window overlaps:** Features and labels must use separate, clearly defined time windows. Using information from the label period in the features would cause data leakage and make model performance appear artificially high.
- **Limited ground truth:** The available daily performance fields do not directly confirm whether a page genuinely needs refreshing. A future performance decline can serve as a provisional proxy, but it does not prove that updating the page will improve results.
- **Observational data:** Performance changes may have many causes, including seasonality, competition, and changes in search demand. The data alone cannot establish that refreshing content caused a change.

Therefore, the score should prioritize pages for human review rather than automatically deciding which pages must be updated.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.